# ObjectGS — End-to-End Colab

Reconstruct a 3D object from a video using SAM2, COLMAP and ObjectGS.

**Workflow:** Setup → Video → Select object → Track → COLMAP → ObjectGS.


## 1. Setup

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/ObjectGS")
REPO_URL = "https://github.com/ArielStr/ObjectGS.git"

if not (REPO / "setup.sh").is_file():
    print("[SETUP] Cloning public ObjectGS repository...")

    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO)],
        check=True,
    )

    print("[SETUP] Repository cloned successfully.")
else:
    print("[SETUP] Repository already exists. Skipping clone.")

# Install dependencies with live Colab output
import html
import time
from IPython.display import display, HTML


def stream_setup_logs(cmd, cwd):
    from collections import deque

    started = time.monotonic()
    log_path = Path(cwd) / "setup_full.log"
    recent = deque(maxlen=60)
    warning_count = 0

    print(f"[SETUP] Starting installation...", flush=True)
    print(f"[SETUP] Full log: {log_path}", flush=True)

    process = subprocess.Popen(
        [str(arg) for arg in cmd],
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors="replace",
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )

    try:
        with log_path.open("w", encoding="utf-8") as log:
            for line in process.stdout:
                log.write(line)
                log.flush()
                recent.append(line.rstrip())

                clean = line.strip()

                if clean.startswith("[ObjectGS]"):
                    print(clean, flush=True)

                elif (
                    clean.startswith("ERROR:")
                    or clean.startswith("Error:")
                    or clean.startswith("E:")
                    or "Traceback (most recent call last)" in clean
                ):
                    print(f"[SETUP ERROR] {clean}", flush=True)

                elif (
                    clean.startswith("WARNING:")
                    or clean.startswith("W:")
                    or clean.startswith("warning:")
                ):
                    warning_count += 1
                    if warning_count <= 5:
                        print(f"[SETUP WARNING] {clean}", flush=True)

    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    finally:
        process.stdout.close()

    code = process.wait()
    elapsed = time.monotonic() - started

    print(f"[SETUP] Elapsed: {elapsed / 60:.1f} minutes")

    if warning_count > 5:
        print(
            f"[SETUP] {warning_count - 5} additional warnings "
            f"saved to {log_path}"
        )

    if code != 0:
        print(f"\n[SETUP FAILED] Exit code: {code}")
        print("\n===== LAST 60 LOG LINES =====")

        for line in recent:
            print(line)

        print(f"\nFull installation log: {log_path}")

        raise subprocess.CalledProcessError(code, cmd)

    print("[SETUP] Installation completed successfully.")
    return code

print("===== OBJECTGS SETUP =====", flush=True)

stream_setup_logs(
    [
        "bash", "setup.sh",
        "--mode", "gpu",
        "--install-colmap",
        "--install-sam2",
    ],
    cwd=REPO,
)

print("ObjectGS setup complete.")


## 2. Configuration

Set the video path, output directory and training parameters.


In [ ]:
from pathlib import Path
import shutil
from datetime import datetime

# Choose "drive" or "upload"
INPUT_SOURCE = "upload"

# Used only when INPUT_SOURCE == "drive"
DRIVE_VIDEO = ""

# Training settings
FPS = 1.0
ITERATIONS = 1000

SAM_CHECKPOINT = REPO / "checkpoints/sam2.1_hiera_small.pt"

# Local workspace for faster processing
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT = Path("/content/objectgs_runs") / RUN_ID
OUTPUT.mkdir(parents=True, exist_ok=False)

input_dir = OUTPUT / "input"
input_dir.mkdir()

if INPUT_SOURCE == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    source_video = Path(DRIVE_VIDEO)
    assert source_video.is_file(), f"Video not found: {source_video}"

elif INPUT_SOURCE == "upload":
    from google.colab import files

    uploaded = files.upload()
    assert len(uploaded) == 1, "Upload exactly one video"
    source_video = Path(next(iter(uploaded)))

else:
    raise ValueError("INPUT_SOURCE must be drive or upload")

VIDEO = input_dir / source_video.name
shutil.copy2(source_video, VIDEO)

assert SAM_CHECKPOINT.is_file(), "SAM2 checkpoint missing"

print("===== OBJECTGS CONFIGURATION =====")
print("Source:", INPUT_SOURCE)
print("Video:", VIDEO)
print("Workspace:", OUTPUT)
print("FPS:", FPS)
print("Iterations:", ITERATIONS)


## 3. Generate object candidates

In [ ]:

import subprocess
import html
from IPython.display import display, HTML

def run_with_live_logs(cmd, cwd):
    """Stream subprocess output directly into Colab."""
    process = subprocess.Popen(
        [str(arg) for arg in cmd],
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors="replace",
        bufsize=1,
    )

    try:
        for line in process.stdout:
            display(HTML(
                "<pre style='margin:0;white-space:pre-wrap'>"
                + html.escape(line.rstrip("\n"))
                + "</pre>"
            ))
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    finally:
        process.stdout.close()

    code = process.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)

    return code

import subprocess

cmd = [
    "python", "-u", str(REPO / "scripts/run_pipeline.py"),
    "--video", str(VIDEO),
    "--output", str(OUTPUT),
    "--sam-checkpoint", str(SAM_CHECKPOINT),
    "--fps", str(FPS),
    "--prepare-selection",
]

run_with_live_logs(cmd, cwd=REPO)


## 4. Select object

In [ ]:
import sys
import importlib

scripts_dir = str(REPO / "scripts")

if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

import colab_mask_selector
importlib.reload(colab_mask_selector)

# Default: interactive bounding-box selection
box_ui = colab_mask_selector.show_box_selector(
    image_path=OUTPUT / "images/frame_0000.jpg",
    candidates_dir=OUTPUT / "selection",
    checkpoint=SAM_CHECKPOINT,
    repo_dir=REPO,
    device="cuda",
)

# Alternative: automatic mask selection
# colab_mask_selector.show_mask_selector(
#     image_path=OUTPUT / "images/frame_0000.jpg",
#     candidates_dir=OUTPUT / "selection",
#     checkpoint=SAM_CHECKPOINT,
#     repo_dir=REPO,
# )


## 5. Run ObjectGS

After selecting the object above, run this cell to continue the pipeline.


In [ ]:
import subprocess

initial_mask = OUTPUT / "selection/final_mask.png"
assert initial_mask.exists(), "Select an object first."

cmd = [
    "python", "-u", str(REPO / "scripts/run_pipeline.py"),
    "--video", str(VIDEO),
    "--output", str(OUTPUT),
    "--sam-checkpoint", str(SAM_CHECKPOINT),
    "--initial-mask", str(initial_mask),
    "--fps", str(FPS),
    "--iterations", str(ITERATIONS),
    "--sift-gpu", "off",
]

run_with_live_logs(cmd, cwd=REPO)


In [ ]:
# Summarize and save training metrics
import importlib.util

script_path = REPO / "scripts/summarize_training.py"

spec = importlib.util.spec_from_file_location(
    "objectgs_training_summary", script_path
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

metrics = module.summarize(OUTPUT)


## 6. Results

In [ ]:
print("Pipeline output:", OUTPUT)

for path in sorted(OUTPUT.iterdir()):
    print(path.name)


## Export Results
Save the complete experiment to Google Drive or download it as a ZIP archive.

In [ ]:
from pathlib import Path
import shutil

# Choose: "drive" or "download"
EXPORT_MODE = "download"

# Destination used when EXPORT_MODE == "drive"
DRIVE_EXPORT_DIR = ""

run_dir = Path(OUTPUT).resolve()
assert run_dir.is_dir(), f"Run directory not found: {run_dir}"
assert any(run_dir.iterdir()), "Run directory is empty"

if EXPORT_MODE == "drive":
    from google.colab import drive

    drive.mount("/content/drive")

    destination_root = Path(DRIVE_EXPORT_DIR)
    destination_root.mkdir(parents=True, exist_ok=True)

    destination = destination_root / run_dir.name

    if destination.exists():
        raise FileExistsError(
            f"Export already exists: {destination}"
        )

    shutil.copytree(run_dir, destination)

    print("===== DRIVE EXPORT COMPLETE =====")
    print("Saved to:", destination)

elif EXPORT_MODE == "download":
    from google.colab import files

    archive = shutil.make_archive(
        str(Path("/content") / run_dir.name),
        "zip",
        root_dir=run_dir.parent,
        base_dir=run_dir.name,
    )

    print("===== ZIP CREATED =====")
    print("Archive:", archive)
    print("Size (MB):", round(Path(archive).stat().st_size / 1e6, 2))

    files.download(archive)

else:
    raise ValueError("EXPORT_MODE must be drive or download")
